In [1]:
import torch
from torchvision.models import mobilenet_v3_large as mobilenet
from torchvision.models import MobileNet_V3_Large_Weights as pre_weights

from sklearn.metrics import confusion_matrix, accuracy_score

import helpers.data_handler as data_handler
import helpers.utils as utils

### System details

In [2]:
print(f"PyTorch version: {torch.__version__}")

print("--------------------------------------------------")
print(f"Using cuda: {torch.cuda.is_available()}")
print(f"Cuda device: {torch.cuda.get_device_name(torch.cuda.current_device())}")

PyTorch version: 2.2.0+cu118
--------------------------------------------------
Using cuda: True
Cuda device: NVIDIA GeForce GTX 1660 SUPER


### Getting data

In [3]:
NUM_CLASSES = 2

train, validation, test = data_handler.get_datasets()
print(f"Train dataset size: {len(train)}")
print(f"validation dataset size: {len(validation)}")
print(f"test dataset size: {len(test)}")

Train dataset size: 174817
validation dataset size: 96811
test dataset size: 424223


In [4]:
train_loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(validation, batch_size=2500, shuffle=False)
test_loader = torch.utils.data.DataLoader(test, batch_size=2500, shuffle=False)

### Fine tunning

In [11]:
model = mobilenet(weights=pre_weights.IMAGENET1K_V2)
model.classifier[-1] = torch.nn.Linear(1280, NUM_CLASSES)

print(model.classifier)

bce_loss = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=2, bias=True)
)


In [10]:
NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        preds = model(inputs)
        loss = bce_loss(preds.squeeze(1), labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
    
    epoch_loss = running_loss / len(train)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}], Training Loss: {epoch_loss:.4f}")

    # Validation 
    model.eval()

    val_labels = []
    val_preds = []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            preds = model(inputs)

            _, predicted = torch.max(preds, 1)
            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(predicted.cpu().numpy())

    accuracy = accuracy_score(val_labels, val_preds)
    cm = confusion_matrix(val_labels, val_preds)
    
    print(f'Validation Accuracy: {accuracy:.2f}%')
    print("\n")
    utils.print_confusion_matrix(cm)


ValueError: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 2])) is deprecated. Please ensure they have the same size.

### Results

In [ ]:
model.eval()

test_labels = []
test_preds = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = model(inputs)

        _, predicted = torch.max(preds, 1)
        test_labels.extend(labels.cpu().numpy())
        test_preds.extend(predicted.cpu().numpy())

accuracy = accuracy_score(test_labels, test_preds)
cm = confusion_matrix(test_labels, test_preds)

print(f'Test Accuracy: {accuracy:.2f}%')
print("\n")
utils.print_confusion_matrix(cm)